# POPs pesticide sites: hazard screening and the child risk index

A demonstration notebook of the pilot project. It reproduces two methods of the demo stand on real data:

- **the screening model** estimates whether strong contamination of a site is likely, before laboratory analysis;
- **the child risk index** follows the logic of UNICEF's Children's Climate and Environmental Risk Index (CCRI-DRM):
  risk = hazard × children nearby × exposure pathways.

**Contents**

1. Where the data come from and what they contain.
2. How contaminated the sites are.
3. The screening model: features, spatial validation, selection curve, test against chance.
4. What drives the model score.
5. Assessing a site without analyses, and "what if" questions.
6. The child risk index and its robustness to the choice of weights.
7. The plan for the next sampling campaign.

**How to read.** The code is hidden; the "Show code" button at the top of the page reveals it. All calculations run
locally from the layer files in `source_data/chemicals_tj/pop_layer/`, with no database and no internet.
Site names are transliterated from the Russian originals.

In [ ]:
import sys, json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import demo_utils as U
from demo_utils import f, pct
U.setup('en')
sys.path.insert(0, str(U.ROOT / 'db')); sys.path.insert(0, str(U.ROOT / 'webapp'))
import pop_screening_model as M
import pop_i18n                                  # English labels for the stand's values
from i18n import name_en, value_en
LAYER = U.DATA / 'source_data/chemicals_tj/pop_layer'
sites = pd.read_csv(LAYER / 'pop_sites.csv', encoding='utf-8-sig', low_memory=False)
samples = pd.read_csv(LAYER / 'pop_samples.csv', encoding='utf-8-sig', low_memory=False)
risk = pd.read_csv(LAYER / 'child_risk/child_risk_index.csv', encoding='utf-8-sig')
T, L, targets = M.get_data()
y = targets[M.FINAL_TARGET]
print(f"Sites on the map: {len(T)}, with DDT/HCH analyses: {len(L)}, strongly contaminated (≥100 mg/kg): {y.sum()}")

## 1. Where the data come from

The client has no structured tables. The layer was assembled from the chemicals.tj portal: the site map, survey forms,
card text and attached documents (laboratory protocols, sampling logs, scanned reports).
Values were extracted from PDFs and checked against the scans (229 of 233 checked sets matched).

In [ ]:
src = pd.read_csv(LAYER / 'pop_sources.csv', encoding='utf-8-sig')
names = {'AFI126_2019': 'Agrophysical Research Institute, protocol No. 126 (2019, scan)',
         'AFI8_2021': 'Agrophysical Research Institute, protocol No. 8 (2020)',
         'ILIM42_2017': 'ILIM research association, report No. 42 (2016)', 'JOURNAL': 'sampling logs and registers',
         'KUKANIEV_2012': 'Institute of Chemistry, table of 2012', 'NIKITIN_2017': 'Institute of Chemistry, letter of 2017 (scan)'}
t = pd.DataFrame({'Source': src.source_id.map(names), 'Values in the consolidated sample': src.rank_1_2,
                  'Sites': src.n_sites}).sort_values('Values in the consolidated sample', ascending=False)
U.table(t)
U.note(f"Total: {len(sites)} sites (192 on the map and {int((sites.on_map != True).sum())} only in protocols), "
       f"{len(samples)} samples, {f(src.rank_1_2.sum(), 0)} concentration values without duplicates.")

In [ ]:
R = risk.copy()
bins = [-1, .1, 10, 100, 1e9]; labs = ['≤ 0.1 (standard)', '0.1–10', '10–100', '≥ 100 mg/kg']
R['cls'] = np.where(R.burden.isna(), 'no analyses', pd.cut(R.burden, bins, labels=labs).astype(str))
colors = {'no analyses': '#ffffff', '≤ 0.1 (standard)': U.GREEN, '0.1–10': '#e3b23c', '10–100': U.ORANGE, '≥ 100 mg/kg': U.RED}
fig, axs = plt.subplots(1, 2, figsize=(11, 5.2), gridspec_kw={'width_ratios': [1, 1.25]})
for ax, (b, title) in zip(axs, [((67.3, 36.6, 71.2, 41.1), 'All map sites'),
                               ((67.8, 37.1, 69.9, 38.4), 'South of Khatlon region (zoomed in)')]):
    U.hillshade(ax, b)
    for k, c in colors.items():
        d = R[R.cls == k]
        ax.scatter(d.longitude, d.latitude, s=28 if k != '≥ 100 mg/kg' else 40, c=c, edgecolor='#333', lw=.5,
                   label=f'{k} ({len(d)})', zorder=3)
    U.map_axes(ax); ax.set_title(title)
h, l = axs[0].get_legend_handles_labels()
fig.legend(h, l, title='DDT group or HCH, maximum over samples, mg/kg', fontsize=9, title_fontsize=9,
           loc='lower center', ncol=5, bbox_to_anchor=(.5, -.06))
plt.show()

## 2. How contaminated the sites are

The site contamination indicator is the maximum over samples of two totals, DDT+DDD+DDE and HCH isomers, in mg/kg.
The soil standard for DDT is 0.1 mg/kg. The scale is logarithmic: each tick is ten times the previous one.

In [ ]:
b = L.burden.clip(lower=.001)
fig, ax = plt.subplots(figsize=(8, 3.8))
ax.hist(np.log10(b), bins=np.arange(-3, 4.01, .25), color=U.GREY, edgecolor='white')
ax.hist(np.log10(b[b >= 100]), bins=np.arange(-3, 4.01, .25), color=U.RED, edgecolor='white', label='strongly contaminated (≥100 mg/kg)')
for v, lab in [(.1, 'standard 0.1'), (10, '×100'), (100, '×1000')]:
    ax.axvline(np.log10(v), color='#333', ls='--', lw=1); ax.text(np.log10(v) + .05, ax.get_ylim()[1]*.92, lab, fontsize=9)
ax.set_xticks(range(-3, 5), ['0.001', '0.01', '0.1', '1', '10', '100', '1,000', '10,000'])
ax.annotate('not detected', (-2.87, 4.3), ha='center', fontsize=8, color='#555')
ax.set_xlabel('maximum over site samples, mg/kg'); ax.set_ylabel('sites'); ax.legend(loc='upper right')
ax.set_title(f'Contamination of {len(L)} sampled sites'); plt.show()
U.note(f"The standard is exceeded at {int((L.burden > .1).sum())} of {len(L)} sites, 100-fold at {int((L.burden >= 10).sum())}, "
       f"1000-fold at {int((L.burden >= 100).sum())}. The maximum is {f(L.burden.max(), 0)} mg/kg.")

## 3. The screening model

**Question:** can strongly contaminated sites (≥100 mg/kg, 1000 times the standard) be distinguished from the rest
using information available before laboratory analysis?

**Features**, only what is known before analysis:

- the chemicals.tj survey form (area, odour, pit, access, soil cover, crops, animals, water, residents);
- location: region and coordinates;
- contamination of the five nearest sampled sites;
- open data from local rasters: soil pH, elevation and slope, population, flow accumulation, temperature and precipitation.

**Algorithm:** regularised logistic regression, a simple and explainable model.

**Spatial validation.** Neighbouring sites resemble each other, so ordinary cross-validation overstates performance.
Here the sites are split into 10 geographic clusters, and each cluster in turn is excluded from training entirely.
Below is one of the ten splits.

In [ ]:
from sklearn.cluster import KMeans
gr = KMeans(M.N_CLUSTERS, n_init=5, random_state=0).fit_predict(L[['latitude', 'longitude']].values)
fig, ax = plt.subplots(figsize=(7.5, 5))
U.hillshade(ax, (67.3, 36.6, 71.2, 41.1))
ax.scatter(L.longitude, L.latitude, c=gr, cmap='tab10', s=30, edgecolor='#333', lw=.4, zorder=3)
U.map_axes(ax); ax.set_title('10 spatial clusters: the model is tested on a cluster it has not seen')
plt.show()

In [ ]:
from sklearn.metrics import roc_auc_score
FS = "G. Анкета + регион/координаты + соседи + ковариаты"      # key of the final feature set in the stand code
LOGREG = "Логистическая регрессия"                                # key of the logistic regression model
cats, nums = M.FEATURE_SETS[FS]
oof = M.spatial_oof(L, y, cats, nums, LOGREG)     # 10 splits × 10 clusters
aucs = [roc_auc_score(y, o) for o in oof]
score = oof.mean(0); auc = roc_auc_score(y, score)
U.note(f"Out-of-sample AUC: {f(auc, 2)} (across the ten splits from {f(min(aucs), 2)} to {f(max(aucs), 2)}). "
       f"AUC = 0.5 is guessing, 1 is perfect separation.")

In [ ]:
order = np.argsort(-score, kind='stable'); found = np.cumsum(y[order]); k = np.arange(1, len(y) + 1)
rnd = k * y.mean(); ideal = np.minimum(k, y.sum())
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.plot(k, ideal, color=U.GREY, ls=':', lw=1.2, label='ideal selection')
ax.plot(k, rnd, color=U.GREY, ls='--', lw=1.6, label='random selection')
ax.plot(k, found, color=U.BLUE, lw=2.5, label='model-based selection (out of sample)')
for kk in (20, 30):
    ax.plot(kk, found[kk-1], 'o', color=U.BLUE)
    ax.annotate(f'{kk} sites: {found[kk-1]} versus {f(rnd[kk-1], 1)} with random selection', (kk, found[kk-1]),
                xytext=(60, 9 if kk == 30 else 3), textcoords='data', fontsize=9,
                arrowprops=dict(arrowstyle='-', color='#777'))
ax.set_xlabel('sites surveyed (in decreasing order of model score)'); ax.set_ylabel('strongly contaminated sites found')
ax.set_title('Selection curve: the model finds strongly contaminated sites twice as fast')
ax.legend(loc='upper center', bbox_to_anchor=(.5, -.16), ncol=3)
plt.show()

### Test against chance

Could such a result arise by chance? The "contaminated / not contaminated" labels are shuffled and the whole
validation procedure is repeated. If the model found nothing, its AUC would look like that of shuffled data. For the
demonstration, 40 permutations are run (3 splits each); the full pilot calculation used 304 permutations, p = 0.003.

In [ ]:
rng = np.random.default_rng(1); M.N_REPEATS = 3
obs = roc_auc_score(y, M.spatial_oof(L, y, cats, nums, LOGREG).mean(0))
null = [roc_auc_score(yp, M.spatial_oof(L, yp, cats, nums, LOGREG).mean(0))
        for yp in (rng.permutation(y) for _ in range(40))]
M.N_REPEATS = 10
fig, ax = plt.subplots(figsize=(7, 3.4))
ax.hist(null, bins=12, color=U.GREY, edgecolor='white', label='shuffled labels')
ax.axvline(obs, color=U.BLUE, lw=2.5, label=f'real data: {f(obs, 2)}')
ax.set_xlabel('AUC'); ax.set_ylabel('permutations'); ax.legend(); U.comma_axis(ax, 'x', 2)
ax.set_title('Model result versus chance'); plt.show()
U.note(f"None of the 40 permutations reached the AUC of the real data (maximum among permutations: {f(max(null), 2)}). "
       if max(null) < obs else f"{sum(n >= obs for n in null)} of 40 permutations reached the AUC of the real data.")

### Which information is useful

The model was tested with several feature sets (results of the full pilot run, same validation).

In [ ]:
m = pd.read_csv(LAYER / 'model/metrics.csv', encoding='utf-8-sig')
m = m[m.target.str.contains('основная') & (m.model == LOGREG)].copy()     # 'основная' marks the main target
lab = {'A. Анкета': 'survey form', 'B. Анкета + текст карточки': 'survey form + card text',
       'E. Только регион/координаты + соседи': 'location + neighbouring sites',
       'F. Открытые ковариаты (pH, рельеф, население, сток, климат)': 'open data (pH, terrain, population, flow, climate)',
       'G. Анкета + регион/координаты + соседи + ковариаты': 'all together (final model)'}
m = m[m.features.isin(lab)].assign(name=lambda d: d.features.map(lab)).sort_values('auc_mean')
fig, ax = plt.subplots(figsize=(8, 3.4))
cols = [U.BLUE if n.startswith('all') else U.GREY for n in m.name]
ax.barh(m.name, m.auc_mean, color=cols, xerr=[m.auc_mean - m.auc_p05, m.auc_p95 - m.auc_mean], ecolor='#555', capsize=3)
for i, v in enumerate(m.auc_mean): ax.text(.505, i, f(v, 2), va='center', color='white', fontsize=9, fontweight='bold')
ax.axvline(.5, color='#333', lw=.8); ax.set_xlim(.5, .8); ax.set_xlabel('out-of-sample AUC'); U.comma_axis(ax, 'x', 2)
ax.set_title('Contribution of information groups'); plt.show()
U.note("Open data on the terrain alone work as well as the survey form. Card text reduces performance "
       "(too many features for 169 sites), so it is not used in the model. The final set was chosen in advance and "
       "was not re-selected after the recalculation, so that the choice is not fitted to the result.")

## 4. What drives the model score

Logistic regression coefficients: a positive one raises the score, a negative one lowers it.
This is an association in the data, not a cause: for example, "easy access" accompanies old abandoned warehouses.

In [ ]:
NUM = {'ph': 'soil pH', 'elev_m': 'elevation', 'slope_deg': 'slope', 'pop_500m': 'population within 500 m',
       'flowacc_max': 'flow accumulation', 'bio1_temp_c': 'mean annual temperature', 'bio12_precip_mm': 'annual precipitation',
       'nb_mean_log_burden': 'contamination of neighbouring sites', 'nb_max_log_burden': 'maximum at neighbouring sites',
       'nb_dist_km': 'distance to nearest sampled site', 'latitude': 'latitude', 'longitude': 'longitude',
       'square_ord': 'site area', 'depth': 'pit depth', 'lived_on_site': 'living on the site', 'lived_50m': 'residents within 50 m',
       'lived_100m': 'residents within 100 m', 'lived_500m': 'residents within 500 m', 'river_distance_ord': 'distance to river',
       'groundwater_depth_ord': 'groundwater depth', 'well_distance_ord': 'distance to well'}
CAT = {'access': 'access', 'status': 'status', 'agriculture_near': 'crops', 'land': 'land use',
       'animals_food': 'animals and food', 'surface_water': 'surface water', 'soil_cover': 'cover',
       'water': 'water body', 'smell': 'odour', 'pit_dug': 'pit', 'region': 'region', 'flood_zone': 'flood zone'}
def label(n):
    k, v = n.split('__', 1)
    if k == 'num':
        if v.startswith('missingindicator_'): return 'no data: ' + NUM.get(v[17:], v[17:])
        return NUM.get(v, v)
    for c in sorted(CAT, key=len, reverse=True):
        if v.startswith(c + '_'):
            val = value_en(v[len(c)+1:]).replace('Food and edible produce', 'food')
            return U.short(f"{CAT[c]}: {val.lower()}", 52)
    return v
fi = pd.read_csv(LAYER / 'model/feature_importance.csv', encoding='utf-8-sig')
fi = fi[~fi.feature.str.contains('missingindicator|нет данных')]
fi = fi.reindex(fi.weight.abs().sort_values(ascending=False).index).head(10)[::-1]
fig, ax = plt.subplots(figsize=(8.5, 4.6))
ax.barh([label(n) for n in fi.feature], fi.weight, color=[U.BLUE if w > 0 else U.GREY for w in fi.weight])
ax.axvline(0, color='#333', lw=.8); ax.set_xlabel('coefficient (standardised features)'); U.comma_axis(ax, 'x', 1)
ax.set_title('The 10 strongest features: blue raise the score, grey lower it'); plt.show()

## 5. Assessing a site without analyses, and "what if" questions

The final model is trained on all 169 sampled sites and scores the rest. The score is converted to a 0–1 scale
(the share of sampled sites with a lower score) and to a class: bottom, middle or top third. This is how the stand's
"Assess a new site" form works.

In [ ]:
Ltr = M.add_lag(L, L); Tall = M.add_lag(L, T)
model = M.make_model(LOGREG, cats, nums).fit(Ltr[cats + nums], y)
ref = np.sort(model.predict_proba(Ltr[cats + nums])[:, 1])
r01 = lambda p: np.searchsorted(ref, p) / len(ref)
q1, q2 = np.quantile(M.rank01(score), [1/3, 2/3])
cls = lambda s: 'low' if s <= q1 else ('medium' if s <= q2 else 'high')
oof_cls = pd.cut(np.round(M.rank01(score), 3), [-1, q1, q2, 2], labels=['low', 'medium', 'high'])
share = (pd.Series(L.burden.values >= 100).groupby(oof_cls).mean()).to_dict()

un = Tall[T.burden.isna().values].copy()
un['score'] = r01(model.predict_proba(un[cats + nums])[:, 1]); un['class'] = un.score.map(cls)
un = un.merge(risk[['map_id', 'children_1000m']], on='map_id', how='left').sort_values('score', ascending=False)
t = pd.DataFrame({'Code': un.tj_code, 'Site': un.site_name.map(lambda n: U.short(name_en(n), 58)),
                  'Region': un.region.map(value_en), 'Score 0–1': un.score,
                  'Class': un['class'], 'Children within 1 km': un.children_1000m})
U.table(t.head(10), nd={'Score 0–1': 2, 'Children within 1 km': 0})
U.note("Share of strongly contaminated sites among sampled sites, by class: " +
       ", ".join(f"{c}: {pct(v)}" for c, v in share.items()) + ". This is what the class means for a new site.")

In [ ]:
site = un[un.tj_code == 'TJ-7599'].iloc[0] if (un.tj_code == 'TJ-7599').any() else un.iloc[0]
base = Tall[Tall.map_id == site.map_id].copy()
print(f"Example: {site.tj_code}, {name_en(site.site_name)}; access in the survey form: {value_en(base.access.iloc[0])}")
rows = [('as in the survey form', base.access.iloc[0], r01(model.predict_proba(base[cats + nums])[:, 1])[0])]
for a in [v for v in ['Легкий доступ; малое расстояние для входа', 'Контролируемый доступ; вход затруднен',
                      'Удаленные места; менее доступен'] if v != base.access.iloc[0]]:    # survey-form answers as stored
    x = base.copy(); x['access'] = a
    rows.append(('if access: ' + value_en(a).split(';')[0].lower(), a, r01(model.predict_proba(x[cats + nums])[:, 1])[0]))
x = base.copy(); x['nb_mean_log_burden'] = np.log10(0.5)
rows.append(('if neighbouring sites are almost clean (0.5 mg/kg)', '', r01(model.predict_proba(x[cats + nums])[:, 1])[0]))
w = pd.DataFrame(rows, columns=['Variant', 'Access', 'Score 0–1']); w['Class'] = w['Score 0–1'].map(cls)
U.table(w[['Variant', 'Score 0–1', 'Class']], nd=2)
U.note("The inspector sees which information changes the score. This is how the model suggests what to clarify during a site visit.")

## 6. The child risk index

The index answers a different question from the screening model: not "where is it dirtier" but **where intervention
will protect the most children**.

| Component | How it is computed |
|---|---|
| Hazard H | log10(contamination / 0.1) / 4: the standard → 0, 1000 mg/kg → 1; without analyses, the mean H of the screening class |
| Children nearby E | 0.6 × rank of the number of children within 1 km (WorldPop R2025A under-18 age raster, 2025) + 0.25 × rank of the number of people on the site + 0.15 × mention of a school or children |
| Vulnerability V | 0.7 × share of seven exposure pathways (drinking water, irrigation, livestock, crops, free access, river or flood zone within 300 m, groundwater shallower than 10 m) + 0.3 × poverty from the Meta Relative Wealth Index within 2 km |
| Risk | (H × E × V)^(1/3), each component at least 0.05 |

In [ ]:
chk = np.cbrt(risk.H.clip(lower=.05) * risk.E.clip(lower=.05) * risk.V.clip(lower=.05))
print('Recalculation by the formula matches the stand index:', bool(np.allclose(chk.round(3), risk.child_risk, atol=.0015)))
top = risk.sort_values('child_risk', ascending=False).head(10)
U.table(pd.DataFrame({'Rank': top.child_risk_rank, 'Code': top.tj_code, 'Site': top.site_name.map(lambda n: U.short(name_en(n), 55)),
                      'H': top.H, 'E': top.E, 'V': top.V, 'Risk': top.child_risk,
                      'Children within 1 km': top.children_1000m}), nd={'H': 2, 'E': 2, 'V': 2, 'Risk': 2, 'Children within 1 km': 0})
fig, ax = plt.subplots(figsize=(7.5, 4.6))
cc = risk.child_risk_class.map(U.CLASS_COLORS)
ax.scatter(risk.H, risk.E, s=20 + 160*risk.V, c=cc, edgecolor='#333', lw=.4, alpha=.9)
for c, col in U.CLASS_COLORS.items(): ax.scatter([], [], c=col, s=50, label=U.cls(c), edgecolor='#333', lw=.4)
ax.set_xlabel('hazard H'); ax.set_ylabel('children nearby E'); U.comma_axis(ax, 'x', 1); U.comma_axis(ax, 'y', 1)
ax.legend(title='risk class (size: pathways V)', fontsize=8, title_fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1))
ax.set_title('High risk is where contamination, children and exposure pathways coincide'); plt.show()

### Robustness to the choice of weights

The component weights are an expert decision. To show how much the result depends on it, the index is recalculated
1000 times with random weights for H, E and V (a third each on average, spread about ±0.1), and we count how often
each site stays in the top ten.

In [ ]:
rng = np.random.default_rng(0)
lg = np.log(risk[['H', 'E', 'V']].clip(lower=.05).values)
W = rng.dirichlet([20, 20, 20], 1000)
top10 = np.zeros(len(risk))
for w in W:
    s = (lg * w).sum(1); top10[np.argsort(-s)[:10]] += 1
risk['in_top10'] = top10 / len(W)
st = risk[risk.in_top10 > .05].sort_values('in_top10', ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.8))
ax.barh(st.tj_code + ' · ' + st.site_name.map(lambda n: U.short(name_en(n), 36)), 100*st.in_top10,
        color=[U.BLUE if v >= .8 else U.GREY for v in st.in_top10])
for i, v in enumerate(st.in_top10): ax.text(100*v + 1, i, pct(v), va='center', fontsize=8)
ax.set_xlim(0, 112); ax.set_xlabel('% of recalculations in which the site is in the top ten')
ax.set_title('The core of the top ten does not depend on the choice of weights'); plt.show()
U.note(f"{int((risk.in_top10 >= .8).sum())} sites stay in the top ten in at least 80% of recalculations: "
       "intervention priorities are robust.")

## 7. The plan for the next sampling campaign

The model and the index turn into a concrete plan: which sites to survey and why.

In [ ]:
plan = pd.read_csv(LAYER / 'sampling_plan/sampling_plan.csv', encoding='utf-8-sig')
plan['group_en'] = plan.group_name.map(value_en)
summ = json.loads((LAYER / 'sampling_plan/summary.json').read_text(encoding='utf-8'))
g = plan.groupby('group_en', sort=False).agg(Sites=('tj_code', 'size'), Samples=('n_samples', 'sum'),
                                             Children=('children_1000m', 'sum')).reset_index()
g.columns = ['Group', 'Sites', 'Samples', 'Children within 1 km']
U.table(g, nd=0)
fig, ax = plt.subplots(figsize=(7.5, 5))
U.hillshade(ax, (67.3, 36.6, 71.2, 41.1))
ax.scatter(risk.longitude, risk.latitude, s=8, c='#999', zorder=2, label='other sites')
pc = [U.RED, U.ORANGE, U.BLUE, U.GREEN]
for (gn, d), c in zip(plan.groupby('group_en', sort=False), pc):
    ax.scatter(d.longitude, d.latitude, s=55, c=c, edgecolor='#222', lw=.6, zorder=3, label=f'{gn} ({len(d)})')
U.map_axes(ax); ax.legend(fontsize=8, loc='upper left', bbox_to_anchor=(1.01, 1)); ax.set_title('Sampling plan sites by group'); plt.show()
U.note(f"{summ['n_sites']} sites, about {summ['n_samples']} samples. In the initial assessment group the model expects "
       f"about {f(summ['group1_expected_strong'], 0)} strongly contaminated sites; the campaign will be an independent test of the model.")

## Conclusions

- Scattered PDFs and cards were turned into a verifiable data layer: every value refers to its source document.
- Using information available before analysis, the screening model finds strongly contaminated sites about twice as
  fast as random selection; the result was validated on geographically held-out sites and is not explained by chance.
- The model reliably identifies the top third of sites but separates the middle and bottom thirds poorly. It sets the
  order of surveys but does not replace the laboratory.
- The child risk index is transparent, can be recalculated from the formula and is robust to the choice of weights.
- The next step is a campaign following the sampling plan: it will test the model on new sites and provide data to improve it.